In [1]:
import pandas as pd
import os
import chephy_model as cpm


In [2]:
def prepare_data(data, cdr3_header):

    # remove cdr3 sequences that contain non-aa letters (this also removes nan)
    amino_acid_pattern = r'^[ARNDCEQGHILKMFPSTWYV]+$'

    data_filtered = data[data[cdr3_header].str.match(amino_acid_pattern, case=False, na=False)]

    data_filtered = data_filtered[data_filtered[cdr3_header].str.len() >= 8]

    return data_filtered

full_df = pd.read_feather("/dsi/sbm/chen2/raw_data/big_data/serumdetect_tcr_data.2024-09-20.ftr")


def prepare_and_truncate(df, cdr3_header):
    df = prepare_data(df, cdr3_header)
    df = cpm.truncate_sequences(df, cdr3_header, 4, 4)
    return df

cdr3_header = "cdr3"

prepared_df = prepare_and_truncate(full_df, cdr3_header)



In [3]:
# Find cdr3_truncated sequences that appear in more than one sample_name
public_seqs = (
    prepared_df.groupby('cdr3_truncated')['sample_name']
    .nunique()
    .loc[lambda x: x > 1]
    .index
)

# Remove rows with public sequences
prepared_df_no_public = prepared_df[~prepared_df['cdr3_truncated'].isin(public_seqs)]

In [4]:
prepared_df = pd.read_feather("/dsi/sbm/chen2/raw_data/big_data/prepared_df.ftr")

In [5]:
prepared_df.reset_index(drop=True).to_feather("/dsi/sbm/chen2/raw_data/big_data/prepared_df.ftr")

In [6]:
metadata = pd.read_csv("/dsi/sbm/chen2/raw_data/big_data/metadata.tsv", delimiter="\t")

In [7]:
# Count samples per patient in prepared_df
sample_counts = prepared_df['sample_name'].value_counts()

# Filter sample_names with more than 5000 samples
sample_names_5000 = sample_counts[sample_counts > 5000].index

# Filter metadata for these sample_names
metadata_5000 = metadata[metadata['sample_name'].isin(sample_names_5000)]

# Count patients by is_cancer status
patient_counts = metadata_5000['is_cancer'].value_counts()
print(patient_counts)

is_cancer
False    586
True     457
Name: count, dtype: int64


In [8]:
metadata.columns

Index(['sample_name', 'is_cancer', 'age_years', 'Gender', 'race_parsed',
       'stage_parsed', ' productive_templates ', ' productive_rearrangements ',
       'smoking', 'has_benign_nodule', 'has_copd', 'has_heart_disease',
       'median_umi_count', 'total_umi_M', 'Histology', 'VJ_lot'],
      dtype='object')